# MiniMax H3 Turbo · Kaggle T4 x2 一键册

按上传的 `notebook-minimax-h3-2k.ipynb` 的架构重排：**配置 → 环境 → 权重 → 启动**，引擎从 GitHub 仓库运行时拉取（不内嵌 3000 行源码）。

| 项 | 说明 |
|---|---|
| 加速器 | **GPU T4 x2**（必须两块；单卡会在加载时报错） |
| Internet | 打开（拉仓库、装依赖、下权重） |
| 权重 | 约 40 GB，全部落在 `/kaggle/tmp`（会话结束即清空，不占 working 配额） |
| 分辨率 | 180p / 360p / 480p / 540p / 720p + 画幅比例（16:9、9:16、1:1、4:3、21:9） |
| 时长 | 3–20 s，步数预设 4 / 6 / 8 |
| 引擎 | `frontend.py`（内部加载 `backend.py`：双卡张量并行 + INT8 ConvRot） |
| 2K | ⚠️ 这册用仓库里的 H3 引擎，上限 720p；2K 变体（Open-H3-Regenerate-2K）的引擎不在仓库，需要上传的那份 notebook |

**Secret**：优先读 Kaggle Secrets，取不到用 Step 0 里的 `HARDCODED_SECRETS`；H3 全流程用不到 secret，`HF_TOKEN` 留空即可。

**权重来源**：`DeepBeepMeep/MiniMax-H3` @ `5905b204` + `pdmd2026/pdmd_4NFE_lora` @ `9913253b`，如果 `/kaggle/input` 里挂了同名文件会自动复用、不重复下载。


In [ ]:
# ============ 0. 配置（只改这一段） ============
import os, sys, json, math, time, glob, shutil, subprocess

IS_KAGGLE = os.path.isdir("/kaggle")
WORK = "/kaggle/working" if IS_KAGGLE else os.path.abspath("./kaggle_working")
TMP  = "/kaggle/tmp"     if IS_KAGGLE else os.path.abspath("./kaggle_tmp")
os.makedirs(WORK, exist_ok=True)
os.makedirs(TMP, exist_ok=True)

# ---- Secrets：先读 Kaggle Secrets，取不到就用下面的硬编码值 ----
HARDCODED_SECRETS = {
    "HF_TOKEN": "",            # 私有/限流仓库才需要
    "KAGGLE_USERNAME": "",
    "KAGGLE_KEY": "",
    "CF_PRIVATE_HOST": "",
    "CF_TUNNEL_TOKEN": "",
}

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            print(f"🔑 {name}: 来自 Kaggle Secrets")
            return v
    except Exception:
        pass
    v = HARDCODED_SECRETS.get(name, "")
    print(f"🔑 {name}: {'使用硬编码值' if v else '未设置（可选）'}")
    return v

HF_TOKEN = secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
for _k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "CF_PRIVATE_HOST", "CF_TUNNEL_TOKEN"):
    _v = secret(_k)
    if _v:
        os.environ[_k] = _v

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ---- 仓库 / Wan2GP ----
REPO_URL   = "https://github.com/milleniummike-del/kaggle-minimax.git"
REPO_DIR   = os.path.join(WORK, "kaggle-minimax")
WAN2GP_DIR = os.path.join(WORK, "Wan2GP")
WAN2GP_URL = "https://github.com/DeepBeepMeep/Wan2GP.git"
WAN2GP_COMMIT = "b8b18f8114e432eea8f3d7e853a51dd91fa99571"

print("=" * 70)
try:
    smi = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total",
                                   "--format=csv,noheader"], text=True).strip()
    for _i, _l in enumerate(smi.splitlines()):
        print(f"✅ GPU {_i}: {_l}")
    if len(smi.splitlines()) < 2:
        print("⚠️ 本册需要 2 块 GPU：Settings → Accelerator → GPU T4 x2")
except Exception:
    print("⚠️ 没检测到 GPU（下载/环境步骤仍可跑）")
print(f"💾 {TMP} 可用 {shutil.disk_usage(TMP).free / 1e9:.0f} GB")
print("=" * 70)


In [ ]:
# ============ 1. 环境：拉仓库 + 固定 Wan2GP + 装依赖 ============
import os, sys, subprocess

def _run(cmd, **kw):
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=True, **kw)

# 1a. 引擎仓库（每册都从这里取 frontend.py / run_ltx*.py）
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    _run(["git", "-C", REPO_DIR, "fetch", "-q", "--depth", "1", "origin", "HEAD"])
    _run(["git", "-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD"])
else:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    _run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR])
print("✅ 仓库就绪:", REPO_DIR)

# 1b. Wan2GP 固定到引擎适配的 commit（换版本会改变算子/精度路径）
if not os.path.isdir(os.path.join(WAN2GP_DIR, ".git")):
    shutil.rmtree(WAN2GP_DIR, ignore_errors=True)
    os.makedirs(WAN2GP_DIR, exist_ok=True)
    _run(["git", "-C", WAN2GP_DIR, "init", "-q"])
    _run(["git", "-C", WAN2GP_DIR, "remote", "add", "origin", WAN2GP_URL])
_run(["git", "-C", WAN2GP_DIR, "fetch", "-q", "--depth", "1", "origin", WAN2GP_COMMIT])
_run(["git", "-C", WAN2GP_DIR, "reset", "-q", "--hard", "FETCH_HEAD"])
print(f"✅ Wan2GP @ {WAN2GP_COMMIT[:10]}")

# 1c. 依赖：只装 H3 路径真正 import 的（Wan2GP 全量 requirements 里的
#     onnxruntime nightly 已经不在 PyPI，会直接装失败）
print("📦 安装依赖 ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts",
                "--timeout", "120", "--retries", "5",
                "mmgp==3.8.2", "comfy-kitchen==0.2.35", "transformers==4.54.0",
                "diffusers==0.36.0", "gguf==0.17.1", "optimum-quanto",
                "accelerate>=1.1.1", "gradio==5.29.0", "huggingface_hub[hf_xet]",
                "hf_xet>=1.5.2", "sentencepiece", "soundfile",
                "einops", "ftfy", "regex", "imageio",
                "fastapi>=0.100.0", "uvicorn>=0.20.0", "python-multipart"],
               check=True)
print("✅ 依赖装好")


In [ ]:
# ============ 2. 权重 → /kaggle/tmp/h3_models（约 40 GB） ============
# 顺序：/kaggle/input 里挂载的同名文件优先复用 → 否则从 Hugging Face 下到 /kaggle/tmp。
from huggingface_hub import hf_hub_download

H3_REPO   = "DeepBeepMeep/MiniMax-H3"
H3_REV    = "5905b204b04d019c69707d89d49be4c8ae1b134f"
PDMD_REPO = "pdmd2026/pdmd_4NFE_lora"
PDMD_REV  = "9913253b4cee1702671c38637bf3941e1de489f5"

MODELS_DIR = os.path.join(TMP, "h3_models")
LORA_DIR   = os.path.join(TMP, "h3_loras")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(LORA_DIR, exist_ok=True)

# (HF 内路径, 体积 GB, 落盘目录)
FILES = [
    ("MiniMax-H3-FL2VA-pruned_rank8_int8_convrot.safetensors",          21.06, MODELS_DIR),
    ("Qwen3-VL-32B-Instruct/qwen3vl-32B-MiniMax-H3-Q4_K_M.gguf",        14.58, MODELS_DIR),
    ("minimax_h3/minimax_h3_video_vae_int8_convrot.safetensors",         2.81, MODELS_DIR),
    ("MiniMax-H3-audio_vae_fp32.safetensors",                            0.61, MODELS_DIR),
    ("Qwen3-VL-32B-Instruct/config.json",                                0.00, MODELS_DIR),
    ("Qwen3-VL-32B-Instruct/tokenizer.json",                             0.00, MODELS_DIR),
    ("Qwen3-VL-32B-Instruct/tokenizer_config.json",                      0.00, MODELS_DIR),
    ("Qwen3-VL-32B-Instruct/vocab.json",                                 0.00, MODELS_DIR),
    ("Qwen3-VL-32B-Instruct/preprocessor_config.json",                   0.00, MODELS_DIR),
    ("Qwen3-VL-32B-Instruct/chat_template.json",                         0.00, MODELS_DIR),
]
DISTILL = ("pdmd2026/pdmd_4NFE_lora", "lora_model_0.safetensors", LORA_DIR, PDMD_REV, 1.38)

_INDEX = {}
if os.path.isdir("/kaggle/input"):
    for _root, _, _files in os.walk("/kaggle/input"):
        for _f in _files:
            _INDEX.setdefault(_f, os.path.join(_root, _f))

def fetch(repo, filename, dest_dir, revision=None, size=0.0):
    dest = os.path.join(dest_dir, filename)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f"  ✓ 已存在 {filename}")
        return dest
    src = _INDEX.get(os.path.basename(filename))
    if src:
        os.symlink(src, dest)
        print(f"  ⚡ 复用 Dataset {filename}")
        return dest
    print(f"  ⬇️ {filename}" + (f" (~{size:.1f} GB)" if size else ""))
    hf_hub_download(repo_id=repo, filename=filename, revision=revision,
                    local_dir=dest_dir, token=os.environ.get("HF_TOKEN") or None)
    return dest

for _f, _sz, _d in FILES:
    fetch(H3_REPO, _f, _d, H3_REV, _sz)
fetch(*DISTILL)

need = 40.4
free = shutil.disk_usage(TMP).free / 1e9
print(f"\n✅ H3 权重就绪 | /kaggle/tmp 剩余 {free:.0f} GB（全套约 {need:.0f} GB）")


In [ ]:
# ============ 3. 启动引擎（跑完会出现 Cloudflare / Gradio 公网链接） ============
# 这一格会一直占着，日志会实时刷出来；中断内核或点停止即可关服务，重跑本格=重启。
import codecs, re, time
import psutil

ENGINE_DIR = os.path.join(WORK, 'kaggle-minimax')
ENGINE     = os.path.join(os.path.join(WORK, 'kaggle-minimax'), "frontend.py")
EXTRA_ENV  = {"H3_MODELS_DIR": os.path.join(TMP, "h3_models"), "H3_LORA_DIR": os.path.join(TMP, "h3_loras"), "H3_VARIANT": "pdmd"}

assert os.path.isfile(ENGINE), f"引擎不存在：{ENGINE}（先跑 Step 1）"
for _k, _v in list(EXTRA_ENV.items()):
    if _k.endswith("_DIR"):
        assert os.path.exists(_v), f"缺目录：{_v}（先跑 Step 2）"

# 清掉上一轮残留的引擎 / cloudflared，避免重复占显存
for _p in psutil.process_iter(["name", "cmdline"]):
    try:
        _args = _p.info["cmdline"] or []
        _name = (_p.info["name"] or "").lower()
        _hit = any(os.path.basename(a) == os.path.basename(ENGINE) for a in _args[1:3])
        if _p.pid != os.getpid() and (_hit or _name == "cloudflared"):
            _p.kill()
    except Exception:
        pass
time.sleep(1)

ENV = os.environ.copy()
ENV.update({k: str(v) for k, v in EXTRA_ENV.items()})
print(f"🚀 cd {ENGINE_DIR} && python {os.path.basename(ENGINE)}")
sys.stdout.flush()

proc = subprocess.Popen([sys.executable, ENGINE], cwd=ENGINE_DIR, env=ENV,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
pending = ""
try:
    while True:
        chunk = os.read(proc.stdout.fileno(), 65536)
        if not chunk:
            break
        pending += decoder.decode(chunk)
        parts = re.split(r"(?<=[\r\n])", pending)
        pending = parts.pop()
        out = "".join(parts)
        if out:
            sys.stdout.write(out)
            sys.stdout.flush()
    sys.stdout.write(pending)
    rc = proc.wait()
    if rc != 0:
        print(f"\n❌ 引擎退出（code={rc}）。code=-9 通常是主机内存不够。")
except KeyboardInterrupt:
    proc.kill()
    proc.wait()
    print("🛑 已停止。重跑本格即可再启动。")
